In [58]:
import ifcopenshell
import ifcopenshell.api
import ifcopenshell.util
import ifcopenshell.api.root
import ifcopenshell.api.context
import ifcopenshell.api.aggregate

In [59]:
bim_object_lib = ifcopenshell.open("blenderbim-site-library.ifc").by_type("IfcProjectLibrary")[0]
bim_objects = bim_object_lib.Declares[0].RelatedDefinitions
for obj in bim_objects:
    print(obj.get_info())

{'id': 91, 'type': 'IfcBuildingElementProxyType', 'GlobalId': '0FMiZScTPFog7h7dFJ1g95', 'OwnerHistory': None, 'Name': 'Mobile Crane 50T', 'Description': None, 'ApplicableOccurrence': None, 'HasPropertySets': None, 'RepresentationMaps': (#227=IfcRepresentationMap(#226,#208), #302=IfcRepresentationMap(#301,#284)), 'Tag': None, 'ElementType': None, 'PredefinedType': None}
{'id': 27, 'type': 'IfcBuildingElementProxyType', 'GlobalId': '2YQNXdYf18kBzxu3tjWlgr', 'OwnerHistory': None, 'Name': 'Site Shed 3x6m', 'Description': None, 'ApplicableOccurrence': None, 'HasPropertySets': None, 'RepresentationMaps': (#58=IfcRepresentationMap(#57,#43),), 'Tag': None, 'ElementType': None, 'PredefinedType': None}
{'id': 59, 'type': 'IfcBuildingElementProxyType', 'GlobalId': '0Q0hvi_v97dhIU2pPJfAJ5', 'OwnerHistory': None, 'Name': 'Site Shed 3x12m', 'Description': None, 'ApplicableOccurrence': None, 'HasPropertySets': None, 'RepresentationMaps': (#90=IfcRepresentationMap(#89,#75),), 'Tag': None, 'ElementType

In [60]:
model = ifcopenshell.file(schema="IFC4")

In [61]:
project = ifcopenshell.api.root.create_entity(model, ifc_class="IfcProject", name="My Project")

# Set up units - using millimeters
units = model.create_entity("IfcUnitAssignment")
length_unit = model.create_entity("IfcSIUnit", UnitType="LENGTHUNIT", Name="METRE")  # Remove the MILLI prefix
units.Units = [length_unit]
project.UnitsInContext = units

# Set up geometric representation contexts
context = model.create_entity(
    "IfcGeometricRepresentationContext",
    ContextType="Model",
    CoordinateSpaceDimension=3,
    Precision=0.01,
    WorldCoordinateSystem=model.create_entity(
        "IfcAxis2Placement3D",
        Location=model.create_entity("IfcCartesianPoint", Coordinates=(0.0, 0.0, 0.0)),
    ),
)

model_context = model.create_entity(
    "IfcGeometricRepresentationSubContext",
    ContextIdentifier="Body",
    ContextType="Model",
    ParentContext=context,
    TargetView="MODEL_VIEW",
)

# Create a site, building, and storey. Many hierarchies are possible.
site = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSite", name="My Site")
building = ifcopenshell.api.root.create_entity(model, ifc_class="IfcBuilding", name="Building A")
storey = ifcopenshell.api.root.create_entity(model, ifc_class="IfcBuildingStorey", name="Ground Floor")

ifcopenshell.api.aggregate.assign_object(model, relating_object=project, products=[site])
ifcopenshell.api.aggregate.assign_object(model, relating_object=site, products=[building])
ifcopenshell.api.aggregate.assign_object(model, relating_object=building, products=[storey])

#13=IfcRelAggregates('2uRbmftQPBbPObSlcDxiJs',$,$,$,#9,(#10))

In [62]:
# Get the crane type from library
crane_type = bim_objects[0]  # Assuming first object is the crane

In [66]:
# First copy all representation items and their styles
for rep_map in crane_type.RepresentationMaps:
    # Copy all items in the representation
    for item in rep_map.MappedRepresentation.Items:
        # Copy styles if they exist
        if hasattr(item, 'StyledByItem'):
            for styled_item in item.StyledByItem:
                # Copy the style assignment
                model.add(styled_item)
                for style in styled_item.Styles:
                    # Copy the presentation style
                    model.add(style)
                    if hasattr(style, 'Styles'):
                        for substyle in style.Styles:
                            # Copy surface styles and colors
                            model.add(substyle)
                            if hasattr(substyle, 'SurfaceColour'):
                                model.add(substyle.SurfaceColour)

In [67]:
# Copy the crane type
new_crane_type = model.add(crane_type)

In [68]:
# Copy the representation maps and their contents
for rep_map in crane_type.RepresentationMaps:
    model.add(rep_map)
    model.add(rep_map.MappedRepresentation)

# Create crane instance
crane = model.create_entity("IfcBuildingElementProxy", Name="cool crane")
crane.ObjectType = new_crane_type.Name

In [69]:
# Create type relationship
type_relationship = model.create_entity(
    "IfcRelDefinesByType",
    GlobalId=ifcopenshell.guid.new(),
    RelatedObjects=[crane],
    RelatingType=new_crane_type
)

# Create placement for the crane
placement = model.create_entity(
    "IfcLocalPlacement",
    PlacementRelTo=storey.ObjectPlacement,
    RelativePlacement=model.create_entity(
        "IfcAxis2Placement3D",
        Location=model.create_entity(
            "IfcCartesianPoint",
            Coordinates=(0.0, 0.0, 0.0)
        )
    )
)
crane.ObjectPlacement = placement

In [70]:
# Copy representation
if new_crane_type.RepresentationMaps:
    shape = model.create_entity(
        "IfcShapeRepresentation",
        ContextOfItems=model_context,
        RepresentationIdentifier=new_crane_type.RepresentationMaps[0].MappedRepresentation.RepresentationIdentifier,
        RepresentationType=new_crane_type.RepresentationMaps[0].MappedRepresentation.RepresentationType,
    )
    
    # Create mapping
    mapped_item = model.create_entity(
        "IfcMappedItem",
        MappingSource=new_crane_type.RepresentationMaps[0],
        MappingTarget=model.create_entity(
            "IfcCartesianTransformationOperator3D",
            Axis1=None,
            Axis2=None,
            LocalOrigin=model.create_entity(
                "IfcCartesianPoint",
                Coordinates=(0.0, 0.0, 0.0)
            ),
            Scale=1.0,
            Axis3=None
        )
    )
    shape.Items = [mapped_item]
    
    # Create product definition shape
    product_shape = model.create_entity(
        "IfcProductDefinitionShape",
        Representations=[shape]
    )
    crane.Representation = product_shape

In [71]:
# Assign the crane to the storey
ifcopenshell.api.aggregate.assign_object(
    model,
    relating_object=storey,
    products=[crane]
)

#5760=IfcRelAggregates('2uNThbmRv7wA3lmf5Dnls0',#5759,$,$,#10,(#5749))

In [72]:
# Copy materials and their associations
if hasattr(crane_type, 'HasAssociations'):
    for association in crane_type.HasAssociations:
        if association.is_a('IfcRelAssociatesMaterial'):
            # Copy the material
            material = association.RelatingMaterial
            model.add(material)
            
            # Copy material properties
            if material.is_a('IfcMaterial'):
                if hasattr(material, 'HasProperties'):
                    for props in material.HasProperties:
                        model.add(props)
                if hasattr(material, 'HasRepresentation'):
                    model.add(material.HasRepresentation)
                    for rep in material.HasRepresentation.Representations:
                        model.add(rep)
            
            # Copy the association
            model.add(association)

In [73]:
# Write the new file
model.write("test.ifc")